# MIDOG++ count-error region selection - Colab runner

Runs the pipeline defined in this repository on a CUDA runtime. The local machine (Windows,
no GPU) is used for audit, tests and the interface only; see `docs/COMPUTE.md`.

**Before running:** this notebook expects the project code in your Google Drive at
`MyDrive/midogcx_project` and specimen images at `MyDrive/midogpp_data`.
The 55-specimen cohort is 7.52 GiB.

Specimens are staged once per session to `/content/midogpp_data` and read from there:
a region crop still decodes the whole TIFF, so reading from the Drive mount would cost
hours of I/O per training run. Drive holds the dataset as cold storage only.

**Runtime > Run all** works end to end on either device. It picks CUDA when present and
falls back to CPU with a printed cost warning, reuses an existing baseline checkpoint
instead of retraining it, and reuses a finished baseline gate. Expected GPU time: ~1.5 h
(gate ~10 min, five branches ~60 min). Expected CPU time: ~7-8 h.

This notebook produces real experimental output. It does not fabricate or approximate any
number: if a step cannot run, it stops and says why.


In [ ]:
import subprocess, sys, os, shutil, textwrap
smi = shutil.which("nvidia-smi")
if smi:
    print(subprocess.run([smi, "--query-gpu=name,memory.total", "--format=csv"],
                         capture_output=True, text=True).stdout.strip())
else:
    print("nvidia-smi not found: this runtime has no GPU attached.")
    print("Runtime > Change runtime type > T4 GPU, then re-run this cell and the next.")
print("python", sys.version.split()[0])


In [ ]:
# Colab ships CUDA torch already; only add what is missing.
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "opencv-python-headless", "PyYAML", "scikit-learn"], check=True)
import torch, torchvision
print("torch", torch.__version__, "cuda available:", torch.cuda.is_available())

# Run all works on either device. Measured cost: 5.85 s/update on this project's CPU
# against 0.76 s/update on a T4, i.e. ~7 h for the five branches instead of ~30 min.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 4 if DEVICE == "cuda" else 2
if DEVICE == "cuda":
    print("using:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no CUDA device - training will run on CPU.")
    print("  ~50 min per 500-update run, ~7-8 h for the whole notebook.")
    print("  For ~1.5 h instead: Runtime > Change runtime type > T4 GPU, restart, Run all.")
print("DEVICE =", DEVICE, "| BATCH_SIZE =", BATCH_SIZE)


## 1. Project code and paths

If you have not uploaded the project yet, upload the folder as a zip to MyDrive, then set
`PROJECT_DIR` accordingly. The notebook never edits the project; it only runs its scripts.

In [ ]:
PROJECT_DIR = "/content/drive/MyDrive/midogcx_project"
DRIVE_DATA  = "/content/drive/MyDrive/midogpp_data"   # 7.52 GiB, cold storage only
LOCAL_DATA  = "/content/midogpp_data"                 # session disk, used for reading

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    # Not Colab: take the paths from the environment so the notebook still runs.
    PROJECT_DIR = os.environ.get("PROJECT_DIR", os.getcwd())
    DRIVE_DATA = os.environ.get("MIDOG_DATA_ROOT", DRIVE_DATA)
    LOCAL_DATA = DRIVE_DATA
    print("not running in Colab; PROJECT_DIR =", PROJECT_DIR)

# Specimens are read from local disk, never from the Drive mount. Reading a region does
# not avoid decoding the whole ~145 MiB TIFF, so one 500-update run touches ~140 GiB
# (~1000 items x 2 orders of magnitude more than the file itself). Over Drive's FUSE
# mount that dominates training time; on /content it is negligible.
import glob


def _n_tiff(root):
    return len(glob.glob(os.path.join(root, "*.tiff")))


DATA_ROOT = LOCAL_DATA
os.makedirs(LOCAL_DATA, exist_ok=True)
if LOCAL_DATA != DRIVE_DATA and _n_tiff(DRIVE_DATA) == 55 and _n_tiff(LOCAL_DATA) < 55:
    print("staging 55 specimens from Drive to local disk (7.52 GiB, once per session) ...")
    shutil.copytree(DRIVE_DATA, LOCAL_DATA, dirs_exist_ok=True)
    print("staged:", _n_tiff(LOCAL_DATA), "specimens")

os.environ["MIDOG_DATA_ROOT"] = DATA_ROOT
if not os.path.isdir(PROJECT_DIR):
    raise SystemExit(f"project not found at {PROJECT_DIR}: clone or upload the repo, then set PROJECT_DIR")
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
print("project:", os.getcwd())
print("specimens:", DATA_ROOT, "->", _n_tiff(DATA_ROOT), "tiff")
print("free space (GB) - local:", round(shutil.disk_usage(LOCAL_DATA).free / 1e9, 1),
      "| results disk:", round(shutil.disk_usage(PROJECT_DIR).free / 1e9, 1))


In [ ]:
# Sanity: code must pass its own tests before any training time is spent.
subprocess.run([sys.executable, "-m", "pytest", "tests/", "-q"], check=True)


## 2. Dataset, frozen cohort and frozen region grid

Metadata ships with the project. Both frozen artefacts are **verified, never
regenerated**: the split and the region grid define every selection, budget and
previously reported number, so a remote runtime must not be able to rewrite them.


In [ ]:
# Verifies the frozen cohort instead of rewriting it: the split is immutable.
subprocess.run([sys.executable, "scripts/build_cohort.py", "--check"], check=True)


In [ ]:
# Exact download sizes first. Nothing is fetched by --dry-run.
subprocess.run([sys.executable, "scripts/download_specimens.py", "--dry-run"], check=True)


In [ ]:
# The cohort is 7.52 GiB. The downloader refuses to start if the disk cannot hold it,
# resumes partial files, and verifies byte-exact sizes. Already-present files are skipped.
subprocess.run([sys.executable, "scripts/download_specimens.py"], check=True)


## 3. Regions and normalisation statistics

Reads every cohort specimen once: writes the non-overlapping 512x512 region grid, the
physical spacing table, and channel statistics computed **only** from source training
specimens.

In [ ]:
# Verifies the frozen region grid (regions.csv, geometry.csv, normalization.json)
# against a fresh recomputation and writes nothing: selections and budgets are
# defined against this exact grid.
subprocess.run([sys.executable, "scripts/prepare_regions.py", "--check"], check=True)


## 4. Source-domain detector

A real run needs far more than the smoke-test budget. Measure first: the script logs
seconds per update, so a session can be cut short without losing the result. The checkpoint
stores its own architecture, normalisation and update count.

In [ ]:
import glob, os
RETRAIN_BASELINE = False   # True forces a fresh baseline even if a checkpoint exists
existing = sorted(glob.glob("outputs/runs/*/checkpoint.pth"), key=os.path.getmtime)
if existing and not RETRAIN_BASELINE:
    print("reusing existing baseline checkpoint:", existing[-1])
    print("set RETRAIN_BASELINE = True in this cell to train a new one")
else:
    print("training the source baseline on", DEVICE, "at batch", BATCH_SIZE)
    subprocess.run([sys.executable, "scripts/train_detector.py",
                    "--tag", "source_baseline",
                    "--set", "training.device=" + DEVICE,
                    "training.batch_size=" + str(BATCH_SIZE),
                    "training.updates=500"], check=True)


In [ ]:
import glob, os
ckpts = sorted(glob.glob("outputs/runs/*/checkpoint.pth"), key=os.path.getmtime)
print("\n".join(ckpts[-3:]) or "(no checkpoints found)")
if not ckpts:
    raise SystemExit("no checkpoint under outputs/runs/*/checkpoint.pth - run cell 14 first")
SOURCE_CKPT = ckpts[-1]
print("using:", SOURCE_CKPT)


## 5. Acquisition experiment

Threshold calibration, predictor fitting on out-of-sample source specimens, the five
rankings, matched adaptation branches and independent melanoma assessment. A `source_only`
control is always evaluated.

In [ ]:
# GATE: evaluate the unadapted control before spending time on any branch.
# If precision/recall/F1 are near zero here, the acquisition comparison is
# meaningless, so fix the baseline first.
gate_done = glob.glob("outputs/experiments/*baseline_gate/metrics/source_only.json")
if gate_done:
    print("baseline gate already evaluated:", gate_done[-1])
    print("delete that run directory to evaluate it again")
else:
    subprocess.run([sys.executable, "scripts/run_acquisition_experiment.py",
                    "--checkpoint", SOURCE_CKPT,
                    "--tag", "baseline_gate",
                    "--device", DEVICE,
                    "--stop-after-control"], check=True)


In [ ]:
# Five matched adaptation branches plus the source_only control, all on one device.
subprocess.run([sys.executable, "scripts/run_acquisition_experiment.py",
                "--checkpoint", SOURCE_CKPT,
                "--tag", "pilot",
                "--device", DEVICE], check=True)


In [ ]:
import glob, os, json
import pandas as pd
runs = [r for r in sorted(glob.glob("outputs/experiments/*"), key=os.path.getmtime)
        if os.path.exists(os.path.join(r, "comparison.csv"))]
if not runs:
    raise SystemExit("no completed run under outputs/experiments - run cell 17 or 18 first")
run = runs[-1]
print(run)
print(pd.read_csv(os.path.join(run, "comparison.csv")).to_string(index=False))
report = os.path.join(run, "predictor_report.json")
if os.path.exists(report):
    print(json.dumps(json.load(open(report)), indent=2)[:4000])


## 6. Second seed (optional, needed for a variability estimate)

Everything is seeded from `project.seed`; changing it produces an independent replicate.
Report seed variability separately from specimen variability.

In [ ]:
# Independent replicate (report seed variability separately from specimen variability).
# !python scripts/train_detector.py --tag source_seed2 --set project.seed=7 training.device=cuda training.batch_size=4
# !python scripts/run_acquisition_experiment.py --checkpoint outputs/runs/<seed2_run>/checkpoint.pth --tag pilot_seed2 --device cuda --set project.seed=7


## 7. Persist results

Small result files are what you actually need to keep; checkpoints can be regenerated.

In [ ]:
if "run" not in globals():
    raise SystemExit("run cell 19 first: it selects the experiment directory to archive")
ARCHIVE = "/content/drive/MyDrive/midogcx_results"
os.makedirs(ARCHIVE, exist_ok=True)
shutil.make_archive(os.path.join(ARCHIVE, "results_" + os.path.basename(run)), "zip", run)
print("archived:", ARCHIVE)


## What to report

Use `comparison.csv` for the method table and `paired_comparisons.json` for per-specimen
differences with intervals. Always report counting error **together with** precision, recall,
F1 and the raw FP/FN counts — count error can be small while false positives and false
negatives cancel. Do not present any pilot-era number from the research blueprint as a result
of this run.